# Qwen3 4B No-Thinking vs Frugal-Thinking 4B Transformers Benchmark

Purpose: compare `Qwen/Qwen3-4B` in non-thinking mode against `MBZUAI-Paris/Frugal-Thinking-4B` on a few addition problems using Hugging Face Transformers, not vLLM.

The notebook records per-example latency, generated token counts, final parsed number, correctness, and whether the thinking model hit the configured thinking budget and required a forced `</think>` close before asking for a final answer.

## 1. Clone Or Update Repo

This notebook is self-contained, but cloning the repo gives us a stable place to write artifacts and keeps the workflow similar to the other carry-trace Colab notebooks.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/ndalton12/carry-trace.git"  # Change if needed.
BRANCH = "main"
REPO_DIR = Path("/content/carry-trace")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("repo", REPO_DIR)
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

## 2. Install Dependencies

Colab usually already has CUDA PyTorch. This installs current Transformers support for Qwen3 and common analysis packages without installing vLLM.

In [ ]:
!python -m pip install -U pip
!python -m pip install "accelerate>=1.0.0" "pandas>=2.2.0" "transformers>=4.55.2" "huggingface_hub[hf_transfer]>=0.23.0" "sentencepiece>=0.2.0" "protobuf>=5.0.0"

## 3. Runtime Setup

`MBZUAI-Paris/Frugal-Thinking-4B` may require accepting the model terms on Hugging Face. If access fails, accept the model page terms and set `HF_TOKEN` in Colab secrets.

In [ ]:
import gc
import json
import re
import time
from dataclasses import dataclass
from typing import Any

import pandas as pd
import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face")
else:
    print("No HF_TOKEN found; public models may still work")

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
    print("bf16 supported", torch.cuda.is_bf16_supported())

## 4. Benchmark Config

Defaults are intentionally small: 5 problems and 2 thinking budgets, for 15 total generations. Increase `SAMPLE_PROBLEMS` or `FRUGAL_THINKING_BUDGETS` only after the first run confirms the model loads and outputs look sane.

In [ ]:
RUN_NAME = "qwen3_4b_vs_frugal_thinking_transformers"
SEED = 20260625

QWEN_NO_THINK_MODEL_ID = "Qwen/Qwen3-4B"
FRUGAL_THINKING_MODEL_ID = "MBZUAI-Paris/Frugal-Thinking-4B"

NON_THINK_MAX_NEW_TOKENS = 128
FRUGAL_THINKING_BUDGETS = [512, 1024]
THINKING_FINAL_ANSWER_TOKENS = 100

TEMPERATURE = 0.6
TOP_P = 0.95
DO_SAMPLE = True

RUN_DIR = REPO_DIR / "runs" / "qwen_frugal_transformers" / f"{RUN_NAME}-{SEED}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

GENERATION_KWARGS = {
    "do_sample": DO_SAMPLE,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
}

print("run_dir", RUN_DIR)

## 5. Sample Problems

These are deliberately plain standard-format addition prompts. The point is to compare model behavior and runtime, not format ablations. The default set includes three base-10 problems and two base-7 problems, keeping the total run under 20 generations.

In [ ]:
BASE10_SAMPLE_PROBLEMS = [
    {"id": "p01_base10_4d_internal_carry", "base": 10, "a": "1124", "b": "922", "answer": "2046"},
    {"id": "p02_base10_4d_many_carry", "base": 10, "a": "8156", "b": "4890", "answer": "13046"},
    {"id": "p03_base10_8d_length_change", "base": 10, "a": "28348210", "b": "82540223", "answer": "110888433"},
]

BASE7_SAMPLE_PROBLEMS = [
    {"id": "p04_base7_mixed_carry", "base": 7, "a": "1234", "b": "456", "answer": "2023"},
    {"id": "p05_base7_long_chain", "base": 7, "a": "6666", "b": "1", "answer": "10000"},
]

SAMPLE_PROBLEMS = BASE10_SAMPLE_PROBLEMS + BASE7_SAMPLE_PROBLEMS

for row in SAMPLE_PROBLEMS:
    if row["base"] == 10:
        row["prompt"] = f"What is {row['a']} + {row['b']}? Give only the final answer digits with no separators."
    else:
        row["prompt"] = (
            f"In base {row['base']}, what is {row['a']} + {row['b']}? "
            f"Give only the final answer digits in base {row['base']} with no separators."
        )
    row["n_digits"] = max(len(row["a"]), len(row["b"]))

pd.DataFrame(SAMPLE_PROBLEMS)


## 6. Helper Functions

Forced close means: first generate only the thinking budget; if the context still has an unclosed `<think>` block at that cap, append `</think>\nFinal answer: ` and continue for `THINKING_FINAL_ANSWER_TOKENS` more tokens. The full output, including the injected close text and continuation, is retained.

In [ ]:
THINK_OPEN = "<think>"
THINK_CLOSE = "</think>"
FORCE_CLOSE_TEXT = "</think>\nFinal answer: "


def render_prompt(tokenizer: Any, prompt: str, enable_thinking: bool | None) -> str:
    """Render a chat prompt, optionally using Qwen's thinking switch."""
    messages = [{"role": "user", "content": prompt}]
    kwargs = {"tokenize": False, "add_generation_prompt": True}
    if enable_thinking is not None:
        kwargs["enable_thinking"] = enable_thinking
    try:
        return tokenizer.apply_chat_template(messages, **kwargs)
    except TypeError:
        kwargs.pop("enable_thinking", None)
        return tokenizer.apply_chat_template(messages, **kwargs)


def has_unclosed_thinking(text: str) -> bool:
    """Return true when the latest think opening has no matching closing tag."""
    return text.rfind(THINK_OPEN) > text.rfind(THINK_CLOSE)


def parse_last_number(text: str, base: int) -> str | None:
    """Extract the last contiguous digit string valid for the requested base."""
    allowed_digits = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"[:base]
    numbers = re.findall(rf"[{re.escape(allowed_digits)}]+", text.replace(",", ""), flags=re.IGNORECASE)
    return numbers[-1] if numbers else None


def output_preview(text: str, max_chars: int = 700) -> str:
    """Return a compact output preview showing the tail when text is long."""
    if len(text) <= max_chars:
        return text
    return text[: max_chars // 2] + "\n... [truncated] ...\n" + text[-max_chars // 2 :]


def load_model(model_id: str) -> tuple[Any, Any]:
    """Load a tokenizer and causal LM for single-GPU Transformers inference."""
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token_id is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "auto"
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    model.eval()
    return tokenizer, model


def tensorize(tokenizer: Any, text: str, device: torch.device) -> dict[str, Any]:
    """Tokenize text and move tensors to the model device."""
    return tokenizer(text, add_special_tokens=False, return_tensors="pt").to(device)


def encode_continuation(tokenizer: Any, text: str, device: torch.device) -> torch.Tensor:
    """Encode continuation text without special tokens on the target device."""
    return tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].to(device)


def generate_one(
    model_label: str,
    model_id: str,
    tokenizer: Any,
    model: Any,
    problem: dict[str, Any],
    *,
    enable_thinking: bool | None,
    max_new_tokens: int,
    force_close_thinking: bool,
    thinking_final_answer_tokens: int,
    seed: int,
) -> dict[str, Any]:
    """Generate one completion and return timing, parsing, and forced-close metadata."""
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    rendered_prompt = render_prompt(tokenizer, problem["prompt"], enable_thinking)
    inputs = tensorize(tokenizer, rendered_prompt, model.device)
    input_width = int(inputs["input_ids"].shape[-1])
    started = time.perf_counter()

    with torch.no_grad():
        first_outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            **GENERATION_KWARGS,
        )
    first_ids = first_outputs[0, input_width:]
    first_text = tokenizer.decode(first_ids, skip_special_tokens=False)
    hit_cap = int(first_ids.numel()) >= max_new_tokens
    forced_close = False
    continuation_token_count = 0
    decoded_output = first_text
    output_ids = first_ids

    if force_close_thinking and hit_cap and has_unclosed_thinking(rendered_prompt + first_text):
        forced_close = True
        close_ids = encode_continuation(tokenizer, FORCE_CLOSE_TEXT, model.device)
        context_ids = torch.cat([first_outputs[:, :], close_ids], dim=1)
        attention_mask = torch.ones_like(context_ids)
        with torch.no_grad():
            continuation_outputs = model.generate(
                input_ids=context_ids,
                attention_mask=attention_mask,
                max_new_tokens=thinking_final_answer_tokens,
                **GENERATION_KWARGS,
            )
        continuation_ids = continuation_outputs[0, context_ids.shape[-1]:]
        continuation_text = tokenizer.decode(continuation_ids, skip_special_tokens=False)
        decoded_output = first_text + FORCE_CLOSE_TEXT + continuation_text
        output_ids = torch.cat([first_ids, close_ids[0], continuation_ids], dim=0)
        continuation_token_count = int(continuation_ids.numel())

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    latency = time.perf_counter() - started
    parsed_answer = parse_last_number(decoded_output, int(problem["base"]))

    return {
        "model_label": model_label,
        "model_id": model_id,
        "example_id": problem["id"],
        "n_digits": problem["n_digits"],
        "base": problem["base"],
        "a": problem["a"],
        "b": problem["b"],
        "answer": problem["answer"],
        "prompt": problem["prompt"],
        "rendered_prompt": rendered_prompt,
        "enable_thinking": enable_thinking,
        "max_new_tokens": max_new_tokens,
        "force_close_thinking": force_close_thinking,
        "thinking_force_closed": forced_close,
        "hit_first_pass_cap": hit_cap,
        "token_count_input": input_width,
        "token_count_output": int(output_ids.numel()),
        "continuation_token_count": continuation_token_count,
        "latency_seconds": latency,
        "tokens_per_second": int(output_ids.numel()) / latency if latency else None,
        "parsed_answer": parsed_answer,
        "correct": parsed_answer == problem["answer"],
        "decoded_output": decoded_output,
    }


def cleanup_model(model: Any | None) -> None:
    """Release model memory between benchmark conditions."""
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


## 7. Run Qwen3-4B With Thinking Disabled

This uses `enable_thinking=False` in the Qwen chat template and a small final-answer token budget.

In [ ]:
all_results = []

print(f"Loading {QWEN_NO_THINK_MODEL_ID}")
tokenizer, model = load_model(QWEN_NO_THINK_MODEL_ID)
for index, problem in enumerate(SAMPLE_PROBLEMS, start=1):
    row = generate_one(
        "qwen3_4b_no_thinking",
        QWEN_NO_THINK_MODEL_ID,
        tokenizer,
        model,
        problem,
        enable_thinking=False,
        max_new_tokens=NON_THINK_MAX_NEW_TOKENS,
        force_close_thinking=False,
        thinking_final_answer_tokens=0,
        seed=SEED + index,
    )
    all_results.append(row)
    print(
        f"[{index}/{len(SAMPLE_PROBLEMS)}] {problem['id']} "
        f"base={row['base']} latency={row['latency_seconds']:.2f}s tokens={row['token_count_output']} "
        f"correct={row['correct']} parsed={row['parsed_answer']}"
    )
    print("decoded_output_preview:")
    print(output_preview(row["decoded_output"]))
    print()

cleanup_model(model)
del tokenizer

## 8. Run Frugal-Thinking-4B With Thinking Budgets

This sweeps a few thinking budgets. If the model is still inside a `<think>` block at the budget cap, the notebook injects `</think>\nFinal answer: ` and lets it continue for `THINKING_FINAL_ANSWER_TOKENS` tokens.

In [ ]:
print(f"Loading {FRUGAL_THINKING_MODEL_ID}")
tokenizer, model = load_model(FRUGAL_THINKING_MODEL_ID)
total = len(SAMPLE_PROBLEMS) * len(FRUGAL_THINKING_BUDGETS)
completed = 0

for budget in FRUGAL_THINKING_BUDGETS:
    print(f"\n=== Frugal thinking budget: {budget} tokens ===")
    for index, problem in enumerate(SAMPLE_PROBLEMS, start=1):
        completed += 1
        row = generate_one(
            f"frugal_thinking_4b_budget_{budget}",
            FRUGAL_THINKING_MODEL_ID,
            tokenizer,
            model,
            problem,
            enable_thinking=True,
            max_new_tokens=budget,
            force_close_thinking=True,
            thinking_final_answer_tokens=THINKING_FINAL_ANSWER_TOKENS,
            seed=SEED + 10_000 + budget + index,
        )
        all_results.append(row)
        print(
            f"[{completed}/{total}] {problem['id']} "
            f"base={row['base']} latency={row['latency_seconds']:.2f}s tokens={row['token_count_output']} "
            f"forced={row['thinking_force_closed']} correct={row['correct']} "
            f"parsed={row['parsed_answer']}"
        )
        print("decoded_output_preview:")
        print(output_preview(row["decoded_output"]))
        print()

cleanup_model(model)
del tokenizer

## 9. Summary And Artifacts

The CSV is compact. The JSONL preserves full prompts and decoded outputs so the tail after a forced close is inspectable even if Colab truncates display output.

In [ ]:
results_df = pd.DataFrame(all_results)
csv_path = RUN_DIR / "qwen_frugal_transformers_results.csv"
jsonl_path = RUN_DIR / "qwen_frugal_transformers_results.jsonl"
summary_path = RUN_DIR / "qwen_frugal_transformers_summary.csv"

results_df.to_csv(csv_path, index=False)
with jsonl_path.open("w") as handle:
    for row in all_results:
        handle.write(json.dumps(row) + "\n")

summary_df = (
    results_df.groupby(["model_label", "base"])
    .agg(
        examples=("example_id", "count"),
        accuracy=("correct", "mean"),
        forced_close_rate=("thinking_force_closed", "mean"),
        mean_latency_seconds=("latency_seconds", "mean"),
        total_latency_seconds=("latency_seconds", "sum"),
        mean_output_tokens=("token_count_output", "mean"),
        mean_tokens_per_second=("tokens_per_second", "mean"),
    )
    .reset_index()
)
summary_df.to_csv(summary_path, index=False)

print("wrote", csv_path)
print("wrote", jsonl_path)
print("wrote", summary_path)

display(summary_df)
display(
    results_df[
        [
            "model_label",
            "base",
            "example_id",
            "n_digits",
            "latency_seconds",
            "token_count_output",
            "thinking_force_closed",
            "hit_first_pass_cap",
            "parsed_answer",
            "answer",
            "correct",
        ]
    ]
)

## 10. Inspect Full Outputs

Use this cell to inspect failures, forced closes, or suspicious parses.

In [ ]:
for row in all_results:
    print("=" * 100)
    print(f"model_label: {row['model_label']}")
    print(f"example_id: {row['example_id']} base={row['base']} n_digits={row['n_digits']}")
    print(f"prompt: {row['prompt']}")
    print(f"answer: {row['answer']}")
    print(f"latency_seconds: {row['latency_seconds']:.2f}")
    print(f"token_count_input: {row['token_count_input']}")
    print(f"token_count_output: {row['token_count_output']}")
    print(f"hit_first_pass_cap: {row['hit_first_pass_cap']}")
    print(f"thinking_force_closed: {row['thinking_force_closed']}")
    print(f"parsed_answer: {row['parsed_answer']} correct={row['correct']}")
    print("rendered_prompt:")
    print(row["rendered_prompt"])
    print("decoded_output:")
    print(row["decoded_output"])
